In [13]:
import os
import dotenv
dotenv.load_dotenv(override=True)  # override=True ensures updated .env values are always picked up
from langchain_groq import ChatGroq

api = os.getenv("API")

if not api:
    raise EnvironmentError("API key not found. Run 'python setup.py' to set your Groq API key.")

LLM = "llama-3.3-70b-versatile"
tina = ChatGroq(
    model=LLM,
    api_key=api,
    temperature=0.7)

# Quick validation — catches invalid keys immediately instead of failing mid-conversation
try:
    tina.invoke("ping")
    print("✓ Groq API key is valid. Tina is ready.")
except Exception as e:
    if "401" in str(e) or "invalid_api_key" in str(e):
        raise EnvironmentError(
            "Invalid Groq API key.\n"
            "Run 'python setup.py --reset' to update it, then re-run this cell."
        ) from None
    raise


✓ Groq API key is valid. Tina is ready.


In [14]:
import sys

# ──────────────────────────────────────────────
#  FIRST-TIME SETUP CHECK
#  Runs every time but only prints instructions
#  when required credentials are missing.
# ──────────────────────────────────────────────

REQUIRED_VARS = {
    "API": "Your Groq API key  →  https://console.groq.com/keys",
    "GOOGLE_CLIENT_ID": "OAuth2 Client ID from your Google Cloud project",
    "GOOGLE_CLIENT_SECRET": "OAuth2 Client Secret from your Google Cloud project",
}

missing = {k: v for k, v in REQUIRED_VARS.items() if not os.getenv(k)}

if missing:
    print("=" * 65)
    print("  SETUP REQUIRED — Missing credentials detected")
    print("=" * 65)
    print()
    print("Each user must register their OWN credentials.")
    print("Never share or commit your Client ID / Secret.\n")

    if "API" in missing:
        print("── [1] Groq API Key ──────────────────────────────────────────")
        print("  1. Sign up at https://console.groq.com")
        print("  2. Go to API Keys → Create API Key")
        print("  3. Add to your .env file:")
        print("       API=your-groq-api-key\n")

    if "GOOGLE_CLIENT_ID" in missing or "GOOGLE_CLIENT_SECRET" in missing:
        print("── [2] Google Cloud OAuth2 Credentials ───────────────────────")
        print("  Each contributor must create their OWN Google Cloud project:")
        print()
        print("  1. Go to https://console.cloud.google.com/")
        print("  2. Create a NEW project (e.g. 'AgentTina-yourname')")
        print("  3. Enable the APIs you need:")
        print("       - Gmail API        (for send_email)")
        print("       - Google Calendar API  (for manage_calendar)")
        print("  4. Go to APIs & Services → Credentials")
        print("  5. Click 'Create Credentials' → OAuth 2.0 Client ID")
        print("  6. Set Application Type to 'Desktop App'")
        print("  7. Copy the values into your .env file:")
        print()
        print("       GOOGLE_CLIENT_ID=your-client-id")
        print("       GOOGLE_CLIENT_SECRET=your-client-secret")
        print()
        print("  ⚠  Do NOT share your credentials or commit them to git.")
        print("     A .gitignore entry for '.env' is already included.\n")

    print("── .env file location ────────────────────────────────────────")
    print(f"  Create/edit:  {os.path.join(os.getcwd(), '.env')}\n")
    print("  After filling in credentials, re-run this cell.")
    print("=" * 65)
    raise SystemExit("Stopping — please complete setup above before continuing.")
else:
    print("✓ All credentials found. Ready to go!")


✓ All credentials found. Ready to go!


In [15]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from datetime import datetime
import base64
import pickle
import subprocess
from email.mime.text import MIMEText
from pathlib import Path
from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

# Gmail OAuth2 scope
GMAIL_SCOPES = ["https://www.googleapis.com/auth/gmail.send"]
TOKEN_FILE = "gmail_token.pickle"

def _get_gmail_service():
    """Authenticate with Gmail using OAuth2 credentials from .env and return a service object."""
    creds = None

    # Load cached token if it exists
    if Path(TOKEN_FILE).exists():
        with open(TOKEN_FILE, "rb") as f:
            creds = pickle.load(f)

    # Refresh or re-authenticate if needed
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            client_config = {
                "installed": {
                    "client_id": os.getenv("GOOGLE_CLIENT_ID"),
                    "client_secret": os.getenv("GOOGLE_CLIENT_SECRET"),
                    "redirect_uris": ["urn:ietf:wg:oauth:2.0:oob", "http://localhost"],
                    "auth_uri": "https://accounts.google.com/o/oauth2/auth",
                    "token_uri": "https://oauth2.googleapis.com/token",
                }
            }
            flow = InstalledAppFlow.from_client_config(client_config, GMAIL_SCOPES)
            creds = flow.run_local_server(port=0)

        # Cache the token for next time
        with open(TOKEN_FILE, "wb") as f:
            pickle.dump(creds, f)

    return build("gmail", "v1", credentials=creds)


# --- Define Tools ---

@tool
def get_current_time(query: str) -> str:
    """Returns the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@tool
def calculator(expression: str) -> str:
    """Evaluates a basic math expression like '2 + 2' or '10 * 5 / 2'."""
    try:
        result = eval(expression, {"__builtins__": {}})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@tool
def web_search(query: str) -> str:
    """Performs a web search and returns the top result."""
    return f"Top search result for '{query}'"

@tool
def search_file(filename: str) -> str:
    """Searches for a file in the local directory and returns its contents."""
    try:
        with open(filename, 'r') as file:
            return file.read()
    except Exception as e:
        return f"Error: {e}"

@tool
def reverse_text(text: str) -> str:
    """Reverses the given text string."""
    return text[::-1]

@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """Sends a real email via Gmail using your Google OAuth2 credentials stored in .env."""
    try:
        service = _get_gmail_service()
        message = MIMEText(body)
        message["to"] = recipient
        message["subject"] = subject
        raw = base64.urlsafe_b64encode(message.as_bytes()).decode()
        service.users().messages().send(userId="me", body={"raw": raw}).execute()
        return f"Email successfully sent to {recipient} with subject '{subject}'."
    except Exception as e:
        return f"Failed to send email: {e}"

@tool
def automate_task(task_description: str) -> str:
    """Automates a task based on the description provided."""
    return f"Task '{task_description}' has been automated."

@tool
def manage_calendar(event_details: str) -> str:
    """Manages calendar events based on the provided details."""
    return f"Calendar event '{event_details}' has been scheduled."

@tool
def run_command(command: str) -> str:
    """Runs a Windows CMD command on the local machine and returns its output.
    Use this to execute shell commands such as listing files, creating directories,
    running scripts, checking system info, etc.
    Example commands: 'dir', 'echo Hello', 'ipconfig', 'mkdir myfolder'
    """
    try:
        result = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=30
        )
        output = result.stdout.strip()
        error = result.stderr.strip()
        if result.returncode != 0:
            return f"Command failed (exit code {result.returncode}):\n{error or output}"
        return output if output else "(Command executed successfully with no output)"
    except subprocess.TimeoutExpired:
        return "Error: Command timed out after 30 seconds."
    except Exception as e:
        return f"Error running command: {e}"

tools = [get_current_time, calculator, reverse_text, web_search, search_file, send_email, automate_task, manage_calendar, run_command]

# --- Create the Agent ---
agent = create_react_agent(tina, tools)


C:\Users\harik\AppData\Local\Temp\ipykernel_24440\2203571955.py:137: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(tina, tools)


In [16]:
# --- Run the Agent ---

def ask_tina(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content


while True:
    user_input = input("Ask Tina: ")
    if user_input.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break
    response = ask_tina(user_input)
    print(f"Tina: {response}")

Tina: The file "Harish" contains the text: "hi. i am hariksan thulasithas. i am 30 years old. i am an AI engieer"
Tina: The command has been executed successfully, and a new folder named "Newz" has been created in the local directory.
Tina: The reversed text of "bye" is "eyb".
Tina: The text "Goodbye" spelled backwards is "eybdooG".
Tina: You're welcome! It was nice chatting with you. Goodbye!
Goodbye!
